# Laboratorio #7 — Fundamentos de Redes Neuronales y Modelos Neuronales de Secuencia
**Natural Language Processing**  
*Universidad Francisco Marroquín*  
**Autor:** George Albadr  
**Fecha de Entrega:** 30 de Septiembre, 2026  

---

### Propósito y Contexto del Laboratorio
Durante el resto del curso de Procesamiento de Lenguaje Natural trabajaremos con modelos neuronales aplicados al lenguaje. Este laboratorio/investigación tiene como objetivo construir de forma sólida y analítica la base conceptual necesaria sobre **redes neuronales feedforward**, **funciones de activación**, **aprendizaje de representaciones**, y **modelos neuronales de secuencia (RNN, LSTM, GRU)**.



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

print("Entorno de trabajo inicializado correctamente.")
print(f"NumPy versión: {np.__version__}")
print(f"Pandas versión: {pd.__version__}")


Entorno de trabajo inicializado correctamente.
NumPy versión: 2.5.2
Pandas versión: 3.0.5


## 1. La Red Neuronal y sus Capas

Antes de abordar modelos avanzados de lenguaje, es indispensable comprender la unidad fundamental de cómputo y las reglas formales que rigen la interconexión de capas en una arquitectura neuronal.

### 1.1 La Neurona Artificial / Perceptrón y sus Componentes
Inspirada en el modelo biológico formalizado inicialmente por McCulloch y Pitts (1943) y el Perceptrón de Rosenblatt (1958), una **neurona artificial** es una unidad de procesamiento matemático que transforma un vector de entradas multidimensional en un único valor escalar de salida.

Sus componentes estructurales son:
1. **Entradas ($x_1, x_2, \dots, x_n$):** Señales numéricas que provienen directamente de los datos observados (e.g., dimensiones de un embedding de palabra o atributos lingüísticos) o de las salidas de neuronas de una capa precedente. Se expresan vectorialmente como $\mathbf{x} = [x_1, x_2, \dots, x_n]^T \in \mathbb{R}^n$.
2. **Pesos Sinápticos ($w_1, w_2, \dots, w_n$):** Parámetros adaptativos entrenables asociados a cada canal de entrada. Representan la fuerza o susceptibilidad sináptica de la conexión. Un peso positivo ($w_i > 0$) ejerce un efecto excitatorio, mientras que un peso negativo ($w_i < 0$) ejerce un efecto inhibitorio sobre la activación de la neurona. Se agrupan en el vector $\mathbf{w} = [w_1, w_2, \dots, w_n]^T \in \mathbb{R}^n$.
3. **Sesgo o Bias ($b$ o $w_0$):** Término independiente que actúa como un umbral intrínseco. Matemáticamente, permite desplazar horizontalmente la función de activación a lo largo del eje del potencial, garantizando que la frontera de decisión no quede anclada obligatoriamente al origen de coordenadas $\mathbf{0}$. Permite que la neurona se active incluso cuando todas las señales de entrada son cero.
4. **Combinador Lineal (Suma Ponderada):** Calcula el producto interno entre el vector de pesos y el vector de entradas, sumando el sesgo:
   $$z = \sum_{i=1}^n w_i x_i + b = \mathbf{w}^T \mathbf{x} + b$$
5. **Función de Activación ($f$):** Transformación escalar (generalmente no lineal) aplicada al potencial $z$. Determina si la neurona "dispara" y con qué intensidad:
   $$a = f(z) = f(\mathbf{w}^T \mathbf{x} + b)$$
6. **Salida ($y$ o $a$):** Valor resultante de la activación, que se transmite como entrada a neuronas subsiguientes o constituye la predicción final de la red.


In [ ]:
# Visualización del Diagrama Propio de la Neurona Artificial
display(Image(filename="Lab#7/figuras/01_neurona_artificial.png", width=750))


<IPython.core.display.Image object>


<IPython.core.display.Image object>

### 1.2 Capas de la Red y el Concepto de Profundidad
Una red neuronal densa (Feedforward / Multi-Layer Perceptron) organiza sus unidades en capas secuenciales:
* **Capa de Entrada (Input Layer):** No realiza procesamiento algebraico ni contiene parámetros entrenables. Su única función es recibir y distribuir el vector de características original $\mathbf{x} \in \mathbb{R}^{d_{in}}$.
* **Capas Ocultas (Hidden Layers):** Capas intermedias que no interactúan de forma directa con el entorno de entrada ni de salida. Su función es realizar proyecciones y deformaciones no lineales sucesivas del espacio geométrico de los datos, extrayendo representaciones latentes de abstracción creciente.
* **Capa de Salida (Output Layer):** Recibe las representaciones procesadas por la última capa oculta y las mapea al formato de predicción requerido por la tarea (e.g., probabilidades de clase discretas mediante Softmax en clasificación o valores continuos en regresión).

**¿Qué significa que una red sea "profunda" (Deep Neural Network)?**  
Una red se califica formalmente como profunda cuando posee **dos o más capas ocultas** ($L \ge 2$). La relevancia de la profundidad radica en el *Principio de Composición Jerárquica*: mientras que una red somera (de una sola capa oculta) puede aproximar cualquier función continua pero requiere un número exponencialmente grande de neuronas (Teorema de Aproximación Universal), una red profunda distribuye el cómputo en niveles. En NLP, las capas iniciales aprenden a detectar patrones léxicos y morfológicos elementales (prefijos, sufijos), las capas medias capturan dependencias sintácticas y frases gramaticales, y las capas profundas sintetizan conceptos semánticos abstractos, tópicos y relaciones de discurso.

---

### 1.3 Cálculo de Parámetros en una Red de Ejemplo: 3 Entradas, 4 Ocultas, 2 Salidas
Analicemos formalmente una arquitectura densa con:
* $N_{in} = 3$ neuronas de entrada.
* $N_h = 4$ neuronas en una única capa oculta.
* $N_{out} = 2$ neuronas en la capa de salida.

**Deducción detallada paso a paso:**
1. **Capa 1 (Entrada $\to$ Capa Oculta):**
   * Cada una de las 4 neuronas ocultas debe conectarse con cada una de las 3 entradas. La matriz de pesos asociada $W^{(1)}$ tiene dimensiones $4 \times 3$ (o $3 \times 4$ según convención de producto):
     $$\text{Pesos}^{(1)} = N_{in} \times N_h = 3 \times 4 = 12 \text{ pesos}$$
   * Cada una de las 4 neuronas ocultas posee su propio término de sesgo independiente:
     $$\text{Sesgos}^{(1)} = N_h = 4 \text{ sesgos}$$
   * **Subtotal Capa 1:** $12 + 4 = 16$ parámetros.

2. **Capa 2 (Capa Oculta $\to$ Capa de Salida):**
   * Cada una de las 2 neuronas de salida recibe conexiones de las 4 neuronas ocultas. La matriz de pesos $W^{(2)}$ tiene dimensión $2 \times 4$:
     $$\text{Pesos}^{(2)} = N_h \times N_{out} = 4 \times 2 = 8 \text{ pesos}$$
   * Cada una de las 2 neuronas de salida posee un término de sesgo:
     $$\text{Sesgos}^{(2)} = N_{out} = 2 \text{ sesgos}$$
   * **Subtotal Capa 2:** $8 + 2 = 10$ parámetros.

**Total General de la Red:**
$$\text{Pesos Totales} = 12 + 8 = 20$$
$$\text{Sesgos Totales} = 4 + 2 = 6$$
$$\mathbf{\text{Total Parámetros Entrenables}} = 20 + 6 = 26$$

*Fórmula general para cualquier capa densa:*  
$P_l = (N_{l-1} + 1) \times N_l$, donde $N_{l-1}$ son las entradas y $N_l$ las neuronas de la capa.


In [ ]:
# Verificación computacional y visualización de la red 3 -> 4 -> 2
display(Image(filename="Lab#7/figuras/02_arquitectura_mlp_3_4_2.png", width=750))

# Simulación numérica del Forward Pass en NumPy
np.random.seed(42)

# Inicialización de parámetros
W1 = np.random.randn(4, 3) * 0.1  # (4, 3)
b1 = np.zeros((4, 1))             # (4, 1)
W2 = np.random.randn(2, 4) * 0.1  # (2, 4)
b2 = np.zeros((2, 1))             # (2, 1)

# Vector de entrada de prueba (3 features)
x = np.array([[0.5], [-1.2], [2.0]])

# Forward pass con activación ReLU en oculta y lineal en salida
z1 = np.dot(W1, x) + b1
h1 = np.maximum(0, z1)           # ReLU
z2 = np.dot(W2, h1) + b2         # Capa de salida

total_pesos = W1.size + W2.size
total_sesgos = b1.size + b2.size

print("--- VERIFICACIÓN DE PARÁMETROS ---")
print(f"Dimensiones W1: {W1.shape} -> {W1.size} pesos")
print(f"Dimensiones b1: {b1.shape} -> {b1.size} sesgos")
print(f"Dimensiones W2: {W2.shape} -> {W2.size} pesos")
print(f"Dimensiones b2: {b2.shape} -> {b2.size} sesgos")
print(f"Total parámetros calculados: {total_pesos + total_sesgos} (Pesos: {total_pesos}, Sesgos: {total_sesgos})")
print(f"Entrada x: {x.ravel()}")
print(f"Activaciones ocultas h1: {h1.ravel()}")
print(f"Salida z2: {z2.ravel()}")


<IPython.core.display.Image object>
--- VERIFICACIÓN DE PARÁMETROS ---
Dimensiones W1: (4, 3) -> 12 pesos
Dimensiones b1: (4, 1) -> 4 sesgos
Dimensiones W2: (2, 4) -> 8 pesos
Dimensiones b2: (2, 1) -> 2 sesgos
Total parámetros calculados: 26 (Pesos: 20, Sesgos: 6)
Entrada x: [ 0.5 -1.2  2. ]
Activaciones ocultas h1: [0.17096513 0.05742251 0.         0.        ]
Salida z2: [-0.00684982 -0.01551139]


<IPython.core.display.Image object>

### 1.4 La Red Neuronal como Composición de Funciones
Desde el punto de vista del análisis matemático, una red neuronal feedforward con $L$ capas no es más que una **función vectorial compuesta de forma anidada**.

Si definimos la operación de la capa $l$ como una función afín seguida de una no linealidad:
$$\phi_l(\mathbf{h}_{l-1}) = f_l(W^{(l)} \mathbf{h}_{l-1} + \mathbf{b}^{(l)})$$
donde $\mathbf{h}_0 = \mathbf{x}$ es la entrada, entonces la red completa evalúa la composición:
$$\hat{\mathbf{y}} = (\phi_L \circ \phi_{L-1} \circ \dots \circ \phi_2 \circ \phi_1)(\mathbf{x})$$
o explícitamente:
$$\hat{\mathbf{y}} = f_L\left(W^{(L)} f_{L-1}\left(\dots f_1(W^{(1)} \mathbf{x} + \mathbf{b}^{(1)})\dots\right) + \mathbf{b}^{(L)}\right)$$

**Implicación Geométrica:**  
Cada capa actúa como un morfismo en el espacio de características. Una sola función afín únicamente puede rotar, escalar y trasladar el espacio euclidiano. La aplicación intercalada de funciones de activación no lineales permite "doblar", "plegar" y "estirar" el espacio de datos. De esta forma, dos clases que se encuentran entrelazadas de manera no linealmente separable en el espacio de entrada original $\mathbb{R}^{d_{in}}$ se desenredan geométricamente capa tras capa, hasta llegar al espacio latente previo a la salida donde pueden separarse limpiamente mediante un hiperplano lineal.


## 2. Activaciones, Pérdida y Entrenamiento

Una red neuronal únicamente se convierte en una herramienta predictiva poderosa cuando puede aprender relaciones complejas no lineales y calibrar sus parámetros mediante gradientes guiados por una función de pérdida.

### 2.1 Colapso Lineal sin Funciones de Activación No Lineales
Si en todas las capas de una red prescindimos de funciones de activación no lineales (o usamos la función identidad $f(z) = z$), la red entera colapsa matemáticamente en un **modelo lineal mono-capa**, sin importar que tenga 2, 50 o 1000 capas.

**Demostración Formal:**  
Consideremos una red de dos capas lineales:
$$\mathbf{h}_1 = W^{(1)} \mathbf{x} + \mathbf{b}^{(1)}$$
$$\mathbf{h}_2 = W^{(2)} \mathbf{h}_1 + \mathbf{b}^{(2)}$$
Sustituyendo $\mathbf{h}_1$ en la segunda ecuación:
$$\mathbf{h}_2 = W^{(2)} \left( W^{(1)} \mathbf{x} + \mathbf{b}^{(1)} \right) + \mathbf{b}^{(2)} = \left( W^{(2)} W^{(1)} \right) \mathbf{x} + \left( W^{(2)} \mathbf{b}^{(1)} + \mathbf{b}^{(2)} \right)$$
Dado que el producto de dos matrices $W_{eff} = W^{(2)} W^{(1)}$ es simplemente otra matriz lineal, y $\mathbf{b}_{eff} = W^{(2)} \mathbf{b}^{(1)} + \mathbf{b}^{(2)}$ es un vector constante, la ecuación resultante:
$$\mathbf{h}_2 = W_{eff} \mathbf{x} + \mathbf{b}_{eff}$$
es idéntica a la de una red de una sola capa lineal. Por inducción matemática, el producto continuo de $L$ matrices afines equivale a una sola transformación lineal. Sin no linealidades, la red pierde por completo la capacidad de representar fronteras no lineales (como el clásico problema XOR) y no puede aprender las estructuras complejas del lenguaje.


In [ ]:
# Demostración numérica: colapso de 3 capas lineales en una sola matriz equivalente
W1 = np.array([[1.0, 2.0], [3.0, 4.0]])
W2 = np.array([[0.5, -1.0], [2.0, 1.5]])
W3 = np.array([[2.0, 0.0], [-1.0, 3.0]])

b1 = np.array([[0.1], [-0.2]])
b2 = np.array([[0.5], [0.0]])
b3 = np.array([[-0.1], [0.4]])

# Entrada aleatoria
x_test = np.array([[1.5], [-0.5]])

# Pasada capa por capa (red profunda lineal)
h1 = W1 @ x_test + b1
h2 = W2 @ h1 + b2
y_profunda = W3 @ h2 + b3

# Colapso analítico en un único paso W_eff y b_eff
W_eff = W3 @ W2 @ W1
b_eff = W3 @ (W2 @ b1 + b2) + b3
y_colapsada = W_eff @ x_test + b_eff

print(f"Salida evaluada capa por capa:\n{y_profunda}")
print(f"Salida evaluada con W_eff única:\n{y_colapsada}")
print(f"Diferencia máxima absoluta: {np.max(np.abs(y_profunda - y_colapsada)):.2e}")
assert np.allclose(y_profunda, y_colapsada), "¡Las salidas deben coincidir numéricamente!"


Salida evaluada capa por capa:
[[-3.1 ]
 [15.85]]
Salida evaluada con W_eff única:
[[-3.1 ]
 [15.85]]
Diferencia máxima absoluta: 4.44e-16


### 2.2 Descripción de Funciones de Activación: Sigmoide, Tanh, ReLU y Softmax

| Función | Definición Matemática | Rango de Salida | ¿En qué parte de la red suele utilizarse? | Ventajas y Limitaciones Principales |
| :--- | :--- | :--- | :--- | :--- |
| **Sigmoide (Logística)** | $\sigma(z) = \frac{1}{1 + e^{-z}}$ | $(0, 1)$ | Capa de salida en clasificación binaria; compuertas (*gates*) en celdas LSTM y GRU. | **Ventaja:** Mapeo probabilístico acotado. **Limitación:** Desvanecimiento severo del gradiente para $|z| > 4$ (derivada máxima $\sigma'(0)=0.25$); no está centrada en cero. |
| **Tangente Hiperbólica (Tanh)** | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1, 1)$ | Capas ocultas en RNNs simples; cálculo de propuestas de candidatos en celdas recurrentes (LSTM y GRU). | **Ventaja:** Centrada en cero, permitiendo gradientes positivos y negativos que balancean el entrenamiento. **Limitación:** Sufre desvanecimiento del gradiente en regiones de saturación. |
| **ReLU (Rectified Linear Unit)** | $\text{ReLU}(z) = \max(0, z)$ | $[0, \infty)$ | Estándar dominante en capas ocultas de redes feedforward profundas, CNNs y capas densas de Transformers. | **Ventaja:** Cálculo ultrarrápido; gradiente constante 1 para $z > 0$ (no satura hacia arriba, combate el vanishing gradient); induce dispersión (*sparsity*). **Limitación:** Problema de *Dying ReLU* si $z \le 0$. |
| **Softmax** | $\text{Softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}$ | $(0, 1)$ con $\sum_{i=1}^K p_i = 1$ | Exclusivamente en la capa de salida para problemas de clasificación multiclase (e.g., predecir el próximo token sobre el vocabulario $|V|$, o etiquetas POS/NER). | **Ventaja:** Convierte logits arbitrarios en una distribución de probabilidad coherente y normalizada. |


In [ ]:
# Visualización de las Funciones de Activación
display(Image(filename="Lab#7/figuras/03_funciones_activacion.png", width=800))


<IPython.core.display.Image object>


<IPython.core.display.Image object>

### 2.3 Funciones de Pérdida: MSE vs Entropía Cruzada (Cross-Entropy)
Una **función de pérdida** (Loss Function, $\mathcal{L}$) cuantifica la discrepancia o penalización entre la predicción emitida por el modelo $\hat{\mathbf{y}}$ y el valor real u objetivo $\mathbf{y}$. Sirve como la función objetivo escalar cuyo gradiente dirige la actualización de los pesos.

* **Error Cuadrático Medio (MSE):**  
  $$\mathcal{L}_{MSE} = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2$$
  Mide la distancia euclidiana cuadrática. Es la función de pérdida estándar en tareas de **regresión continua** bajo el supuesto probabilístico de ruido gaussiano.
* **Entropía Cruzada (Cross-Entropy Loss):**  
  $$\mathcal{L}_{CE} = - \sum_{k=1}^K y_k \log(\hat{y}_k)$$
  Emana directamente de la Teoría de la Información de Shannon y del principio de **Estimación de Máxima Verosimilitud (MLE)**. Mide la divergencia de información entre la distribución verdadera de etiquetas (vector one-hot $\mathbf{y}$) y la distribución de probabilidades predicha $\hat{\mathbf{y}}$.

**¿Por qué Cross-Entropy es el estándar absoluto en clasificación de texto frente a MSE?**
1. **Penalización asintótica:** Si el modelo predice con alta certidumbre una clase equivocada (e.g., $\hat{y}_{correcto} \to 0$), la pérdida por entropía cruzada tiende asintóticamente a infinito ($-\log(0) \to +\infty$), generando una penalización enérgica. MSE, en cambio, acota el error máximo a 1, tratando el error con excesiva indulgencia.
2. **Superficie de gradiente y no saturación:** Cuando se combina una capa de salida Softmax con MSE, la derivada de la pérdida respecto a los logits incluye el término $\hat{y}(1 - \hat{y})$. Si la red comete un error severo pero está saturada, ese término se anula, deteniendo el aprendizaje (*gradient vanishing* inducido por la pérdida). Por el contrario, la combinación **Softmax + Cross-Entropy** simplifica algebraicamente el gradiente a:
   $$\frac{\partial \mathcal{L}_{CE}}{\partial z_i} = \hat{y}_i - y_i$$
   El gradiente es directamente proporcional a la diferencia lineal entre la probabilidad predicha y la verdad fundamental, garantizando un flujo de gradientes limpio, robusto y rápido.


### 2.4 Descenso de Gradiente y Tasa de Aprendizaje ($\eta$)
El **descenso de gradiente** es el motor de optimización que minimiza la pérdida navegando a través del hiperespacio de parámetros.
* **Intuición Física:** Imaginar un terreno montañoso en medio de una niebla espesa donde solo podemos palpar la inclinación inmediata bajo nuestros pies. El vector gradiente $\nabla_\theta \mathcal{L}$ señala hacia donde el terreno sube con mayor inclinación; para alcanzar el valle (mínimo de error), damos un paso en sentido opuesto:
  $$\theta^{(t+1)} = \theta^{(t)} - \eta \nabla_\theta \mathcal{L}(\theta^{(t)})$$
* **Papel de la Tasa de Aprendizaje (Learning Rate, $\eta$):** Define la longitud del paso.
  * **Si $\eta$ es excesivamente grande:** El paso sobrepasa el valle del mínimo, rebotando entre laderas opuestas y provocando oscilaciones crecientes hasta la divergencia numérica completa (pérdida $\to \infty$ o `NaN`).
  * **Si $\eta$ es excesivamente pequeña:** El progreso hacia el mínimo es exasperantemente lento, requiriendo un costo computacional insostenible y corriendo el riesgo de quedar atrapado en mesetas o puntos de ensilladura (*saddle points*).

---

### 2.5 Backpropagation: La Regla de la Cadena en Redes Multicapa
El algoritmo de **retropropagación (Backpropagation)**, popularizado por Rumelhart, Hinton y Williams (1986), es un método sistemático para calcular de forma analítica y exacta las derivadas parciales de la pérdida respecto a cada peso y sesgo de la red: $\frac{\partial \mathcal{L}}{\partial W^{(l)}}$ y $\frac{\partial \mathcal{L}}{\partial \mathbf{b}^{(l)}}$.

**¿Por qué es indispensable para entrenar redes con varias capas?**  
Si intentáramos estimar los gradientes por diferencias finitas numéricas:
$$\frac{\partial \mathcal{L}}{\partial w_i} \approx \frac{\mathcal{L}(w_i + \epsilon) - \mathcal{L}(w_i)}{\epsilon}$$
necesitaríamos realizar una pasada hacia adelante por cada uno de los parámetros de la red. En modelos con millones de parámetros, un solo paso de gradiente tomaría horas. Backpropagation aprovecha la **Regla de la Cadena del Cálculo Multivariable** dividiendo el cómputo en:
1. Una pasada hacia adelante (*Forward Pass*) donde se computan y almacenan en memoria las activaciones intermedias $\mathbf{z}^{(l)}, \mathbf{h}^{(l)}$.
2. Una pasada hacia atrás (*Backward Pass*) donde el error en la salida se propaga recursivamente de la capa $L$ a la 1 utilizando las activaciones almacenadas.  
Esto permite calcular los gradientes exactos de **todos los parámetros simultáneamente** en un tiempo lineal $O(|Connections|)$, volviendo viable el entrenamiento de redes profundas.

---

### 2.6 Epoch, Batch y Overfitting: Técnicas de Mitigación
* **Epoch:** Una iteración completa en la que el algoritmo de optimización ha procesado la totalidad de los datos del conjunto de entrenamiento.
* **Batch (o Mini-batch):** Subconjunto de $B$ ejemplos procesados simultáneamente para calcular una estimación no sesgada del gradiente verdadero y actualizar los pesos (Mini-batch Gradient Descent). Equilibra la estabilidad del cálculo vectorial en GPU y la velocidad de actualización.
* **Overfitting (Sobreajuste):** Ocurre cuando el modelo memoriza el ruido idiosincrático y los detalles irrelevantes del conjunto de entrenamiento en vez de generalizar patrones lingüísticos subyacentes. Se detecta cuando la pérdida de entrenamiento continúa disminuyendo mientras que la pérdida en el conjunto de validación comienza a ascender sistemáticamente.

**Técnicas Clave para Reducir el Sobreajuste:**
1. **Dropout (Srivastava et al., 2014):** Desactivar aleatoriamente con probabilidad $p$ (e.g., $0.2 - 0.5$) una fracción de neuronas en cada paso de entrenamiento. Esto evita la co-adaptación patológica entre unidades, forzando a cada neurona a aprender representaciones robustas y actuando como un ensamble implícito de sub-redes.
2. **Regularización $L_2$ (Weight Decay):** Sumar a la función de pérdida un término proporcional a la norma cuadrática de los pesos: $\mathcal{L}_{reg} = \mathcal{L} + \frac{\lambda}{2} \sum \|W\|^2$. Penaliza magnitudes de peso desmesuradas, forzando a la red a encontrar soluciones más suaves y con menor varianza.
3. **Parada Temprana (Early Stopping):** Monitorear la pérdida en el conjunto de validación tras cada época y abortar el entrenamiento cuando la métrica de validación deje de mejorar durante un número determinado de épocas de paciencia (*patience*).


In [ ]:
# Visualización del Comportamiento del Descenso de Gradiente con Distintas Tasas
display(Image(filename="Lab#7/figuras/04_perdida_gradiente_convergencia.png", width=850))


<IPython.core.display.Image object>


<IPython.core.display.Image object>

## 3. Representación Aprendida

Uno de los pilares conceptuales del Aprendizaje Profundo es que la red neuronal no solo actúa como un discriminador final, sino que construye internamente sus propias representaciones matemáticas de los datos.

### 3.1 Aprendizaje de Representaciones y Espacios Latentes
En el paradigma clásico de NLP (como en los modelos lineales del Lab #4 o n-gramas del Lab #5), el investigador debía diseñar a mano las características (*feature engineering*: listas de stopwords, patrones léxicos, conteos de prefijos). 

En las redes neuronales, el modelo ejecuta **Aprendizaje de Representaciones (Representation Learning)**. La red recibe datos crudos (tokens o índices) y, a medida que la información fluye por las capas ocultas, genera transformaciones sucesivas en espacios euclidianos continuos llamados **espacios latentes**.
* **¿Qué contienen las activaciones de una capa oculta?**  
  Las activaciones $\mathbf{h}^{(l)}$ forman un vector en el que cada dimensión no representa una palabra discreta aislada, sino un rasgo latente desentrelazado. En NLP, estas coordenadas capturan propiedades morfosintácticas (e.g., si la palabra es un verbo transitivo en pasado), propiedades semánticas (e.g., si el referente es animado o inanimado), y relaciones de discurso (e.g., polaridad afectiva o tono epistolar).

---

### 3.2 Representación One-Hot vs Word Embeddings

| Propiedad | Representación One-Hot | Word Embedding (Vector Denso) |
| :--- | :--- | :--- |
| **Estructura matemática** | Vector binario disperso (*sparse*) de dimensión $|V|$. Un '1' en la posición del índice del token y ceros en las demás. | Vector continuo y denso de dimensión fija reducida $d$ (típicamente $d \in [50, 300]$). |
| **Ortogonalidad** | **Total:** Cualquier par de palabras $w_i \neq w_j$ es ortogonal: $\mathbf{v}_i^T \mathbf{v}_j = 0$. $\cos(\mathbf{v}_i, \mathbf{v}_j) = 0$. | **Métrica semántica:** Palabras con contextos similares comparten orientaciones similares: $\cos(\mathbf{v}_i, \mathbf{v}_j) \approx 1$. |
| **Eficiencia de memoria** | Pésima. Requiere matrices de dimensión $|V| \times |V|$ donde el 99.99% son ceros (*curse of dimensionality*). | Excelente. Espacio continuo compacto, altamente eficiente para cómputo tensorial en GPU. |
| **Captura de similitud** | Nula. "Excelente" es tan distante de "maravilloso" como de "submarino". | Óptima. Basada en la Hipótesis Distribucional (Firth, 1957): las palabras que aparecen en entornos similares tienen significados similares. |

**Limitaciones que resuelven los embeddings:**  
One-hot asume implícitamente que todas las palabras son independientes y equidistantes. Los embeddings rompen esta limitación al proyectar las palabras en una variedad geométrica (*manifold*) donde la distancia euclidiana y la similitud coseno reflejan relaciones de significado real.

---

### 3.3 Aritmética Vectorial y Analogías Semánticas
Una de las propiedades más notables de los espacios de embeddings (demostrada empíricamente por Mikolov et al., 2013 con Word2Vec) es que las relaciones semánticas y gramaticales quedan codificadas como **vectores de desplazamiento lineal constante**:

$$\vec{v}_{\text{rey}} - \vec{v}_{\text{hombre}} + \vec{v}_{\text{mujer}} \approx \vec{v}_{\text{reina}}$$

* **Explicación Matemática:**  
  La diferencia $\vec{v}_{\text{mujer}} - \vec{v}_{\text{hombre}}$ aísla la dimensión latente abstracta correspondiente al concepto de *género femenino*. Al sumar este vector de dirección a $\vec{v}_{\text{rey}}$, nos trasladamos en el espacio latente hacia el concepto análogo de realeza con género femenino, aterrizando en la vecindad inmediata de $\vec{v}_{\text{reina}}$.
* **Analogías Sintácticas y Morfológicas:**  
  Este mismo fenómeno se observa en tiempos verbales:
  $$\vec{v}_{\text{caminar}} - \vec{v}_{\text{caminó}} \approx \vec{v}_{\text{hablar}} - \vec{v}_{\text{habló}}$$
  y en relaciones geográficas/capitales:
  $$\vec{v}_{\text{Francia}} - \vec{v}_{\text{París}} + \vec{v}_{\text{Roma}} \approx \vec{v}_{\text{Italia}}$$

---

### 3.4 Conexión con Laboratorios Anteriores: Ventaja frente a Conteos de N-gramas
En el **Laboratorio #5**, evaluamos modelos de n-gramas estadísticos cuya probabilidad dependía estrictamente del conteo de frecuencias exactas sobre el corpus:
$$P(w_i | w_{i-1}) = \frac{C(w_{i-1}, w_i)}{C(w_{i-1})}$$
* **El Cuello de Botella de los N-gramas:** Si en el corpus aparecía repetidamente la secuencia *"el felino maúlla"*, pero nunca se observó *"el gato maúlla"*, el modelo de bigramas asignaba una probabilidad de cero (o residual tras suavizado) a la segunda oración. Trataba a *"gato"* y *"felino"* como entidades léxicas completamente ajenas e inconexas.
* **La Ventaja Neuronal:** En una red con representaciones aprendidas, como *"gato"* y *"felino"* comparten vecinos contextuales similares, sus embeddings aprendidos convergen a vectores muy próximos: $\mathbf{v}_{\text{gato}} \approx \mathbf{v}_{\text{felino}}$. Al evaluar la probabilidad, la red transfiere naturalmente el conocimiento adquirido con *"felino"* hacia *"gato"*, generalizando con fluidez sobre combinaciones de palabras jamás observadas en el entrenamiento y mitigando drásticamente el problema de la **escasez de datos (data sparsity)**.


In [ ]:
# Visualización del Espacio de Embeddings y Aritmética de Analogías
display(Image(filename="Lab#7/figuras/05_espacio_embeddings_analogias.png", width=750))

# Demostración numérica de similitud coseno y aritmética de analogías
palabras = ["rey", "reina", "hombre", "mujer", "perro", "gato"]
# Simulación de embeddings en 4 dimensiones latentes: [Realeza, Femenino, Canino, Felino]
embeddings = {
    "rey":    np.array([ 0.95, -0.80,  0.02, -0.01]),
    "reina":  np.array([ 0.93,  0.85, -0.01,  0.03]),
    "hombre": np.array([ 0.05, -0.88,  0.01, -0.02]),
    "mujer":  np.array([ 0.04,  0.89, -0.02,  0.01]),
    "perro":  np.array([-0.01,  0.00,  0.92, -0.15]),
    "gato":   np.array([ 0.01,  0.02, -0.18,  0.94])
}

def cos_sim(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print("--- SIMILITUD COSENO ---")
print(f"sim(rey, reina):  {cos_sim(embeddings['rey'], embeddings['reina']):.4f}")
print(f"sim(hombre, mujer): {cos_sim(embeddings['hombre'], embeddings['mujer']):.4f}")
print(f"sim(rey, perro):    {cos_sim(embeddings['rey'], embeddings['perro']):.4f}")

# Operación de analogía: rey - hombre + mujer
v_objetivo = embeddings["rey"] - embeddings["hombre"] + embeddings["mujer"]
print("\n--- EVALUACIÓN DE ANALOGÍA: rey - hombre + mujer ---")
for p, v in embeddings.items():
    print(f"Similitud con '{p}': {cos_sim(v_objetivo, v):.4f}")


<IPython.core.display.Image object>
--- SIMILITUD COSENO ---
sim(rey, reina):  0.1297
sim(hombre, mujer): -0.9947
sim(rey, perro):    0.0090

--- EVALUACIÓN DE ANALOGÍA: rey - hombre + mujer ---
Similitud con 'rey': 0.0695
Similitud con 'reina': 0.9981
Similitud con 'hombre': -0.6776
Similitud con 'mujer': 0.7486
Similitud con 'perro': -0.0172
Similitud con 'gato': 0.0382


<IPython.core.display.Image object>

## 4. Redes Neuronales Recurrentes (RNN)

El lenguaje natural no se presenta como una colección estática de atributos independientes, sino como una **secuencia temporal** en la que el orden sintáctico y el contexto histórico condicionan el significado de cada palabra.

### 4.1 Inadecuación de Redes Feedforward Convencionales para Texto
Las redes feedforward estándar fallan al procesar texto de forma nativa debido a tres limitaciones estructurales:
1. **Longitud Variable de las Oraciones:** Una red feedforward clásica exige un vector de entrada de tamaño fijo $D$. En los textos literarios analizados en el Lab #6 (Jane Austen, Mary Shelley, Bram Stoker), las oraciones variaban desde fragmentos de 4 palabras hasta párrafos de 80 palabras. Concatenar tokens y rellenar con *padding* genera una explosión innecesaria de ceros y fija un límite artificial arbitrario.
2. **Pérdida del Orden Secuencial y Sintaxis:** Métodos como Bag of Words (BoW) o promedios estáticos de vectores destruyen el orden gramatical. Frases con significados opuestos como *"el crítico felicitó al autor porque no era malo"* y *"el crítico no felicitó al autor porque era malo"* generan la misma representación.
3. **Falta de Invarianza Temporal (Compartición de Parámetros):** En una feedforward que recibe palabras concatenadas en posiciones contiguas, la red debe aprender un conjunto independiente de pesos para la palabra *"gato"* si aparece al inicio de la frase y otro conjunto completamente distinto si aparece al final. No existe transferencia temporal del conocimiento.

---

### 4.2 Concepto de Estado Oculto y su Actualización
Una **Red Neuronal Recurrente (RNN)** introduce un bucle de retroalimentación interna que dota a la arquitectura de memoria contextual.

* **Estado Oculto ($\mathbf{h}_t$):**  
  Es un vector de estado dinámico que sintetiza la historia de toda la información observada desde el inicio de la secuencia hasta el paso temporal actual $t$. Actúa como una memoria de trabajo continua.
* **Ecuación de Recurrencia:**  
  En cada paso temporal $t$, el estado oculto se actualiza combinando linealmente el estado oculto previo $\mathbf{h}_{t-1}$ con el vector de entrada actual $\mathbf{x}_t$, aplicando una no linealidad $	anh$:
  $$\mathbf{h}_t = 	anh(W_{hh} \mathbf{h}_{t-1} + W_{xh} \mathbf{x}_t + \mathbf{b}_h)$$
  donde:
  * $\mathbf{x}_t \in \mathbb{R}^{d_x}$: vector embedding del token en el instante $t$.
  * $\mathbf{h}_{t-1} \in \mathbb{R}^{d_h}$: estado oculto del paso anterior ($\mathbf{h}_0 = \mathbf{0}$).
  * $W_{xh} \in \mathbb{R}^{d_h 	imes d_x}$: matriz de pesos de entrada a estado oculto.
  * $W_{hh} \in \mathbb{R}^{d_h 	imes d_h}$: matriz de pesos de transición recurrente estado a estado.
  * $\mathbf{b}_h \in \mathbb{R}^{d_h}$: vector de sesgo recurrente.
* **Compartición de Parámetros (Parameter Sharing):**  
  Las matrices $W_{xh}, W_{hh}$ y $W_{hy}$ son **idénticas en todos los pasos temporales**. Esto permite a la RNN procesar secuencias de cualquier longitud sin que el número de parámetros aumente.

---

### 4.3 RNN Desenrollada en el Tiempo para "el gato duerme"
Para entrenar y visualizar una RNN, se "desenrolla" (*unroll*) a lo largo de los pasos temporales de la secuencia:

* **Paso $t=1$:**  
  * Entrada: $\mathbf{x}_1 = 	ext{Embedding}(	ext{"el"})$.
  * Estado inicial: $\mathbf{h}_0 = \mathbf{0}$.
  * Estado oculto 1: $\mathbf{h}_1 = 	anh(W_{hh} \mathbf{h}_0 + W_{xh} \mathbf{x}_1 + \mathbf{b}_h) = 	anh(W_{xh} \mathbf{x}_1 + \mathbf{b}_h)$.
  * Salida 1: $\hat{\mathbf{y}}_1 = 	ext{Softmax}(W_{hy} \mathbf{h}_1 + \mathbf{b}_y)$ (predicción del siguiente token en el paso 1).
* **Paso $t=2$:**  
  * Entrada: $\mathbf{x}_2 = 	ext{Embedding}(	ext{"gato"})$.
  * Recibe $\mathbf{h}_1$ (que ya almacena la noción de un artículo determinante masculino).
  * Estado oculto 2: $\mathbf{h}_2 = 	anh(W_{hh} \mathbf{h}_1 + W_{xh} \mathbf{x}_2 + \mathbf{b}_h)$.
  * Salida 2: $\hat{\mathbf{y}}_2 = 	ext{Softmax}(W_{hy} \mathbf{h}_2 + \mathbf{b}_y)$.
* **Paso $t=3$:**  
  * Entrada: $\mathbf{x}_3 = 	ext{Embedding}(	ext{"duerme"})$.
  * Recibe $\mathbf{h}_2$ (que almacena el sintagma nominal *"el gato"*).
  * Estado oculto 3: $\mathbf{h}_3 = 	anh(W_{hh} \mathbf{h}_2 + W_{xh} \mathbf{x}_3 + \mathbf{b}_h)$.
  * Salida 3: $\hat{\mathbf{y}}_3 = 	ext{Softmax}(W_{hy} \mathbf{h}_3 + \mathbf{b}_y)$.


In [ ]:
# Visualización de la RNN Desenrollada para la Oración "el gato duerme"
display(Image(filename="Lab#7/figuras/06_rnn_desenrollada_el_gato_duerme.png", width=850))

# Simulación en NumPy de la ejecución paso a paso de la RNN desenrollada
np.random.seed(123)

d_x = 4   # Dimensión de embedding
d_h = 3   # Dimensión de estado oculto
d_y = 5   # Vocabulario de salida pequeño

W_xh = np.random.randn(d_h, d_x) * 0.5
W_hh = np.random.randn(d_h, d_h) * 0.5
W_hy = np.random.randn(d_y, d_h) * 0.5
b_h  = np.zeros((d_h, 1))
b_y  = np.zeros((d_y, 1))

tokens = ["el", "gato", "duerme"]
# Vectores simulados de entrada
x_seq = [np.random.randn(d_x, 1) for _ in tokens]

h = np.zeros((d_h, 1))  # h_0
print("--- EJECUCIÓN PASO A PASO RNN ---")
for t, (tok, x_t) in enumerate(zip(tokens, x_seq), start=1):
    h = np.tanh(W_hh @ h + W_xh @ x_t + b_h)
    logits = W_hy @ h + b_y
    probs = np.exp(logits) / np.sum(np.exp(logits))
    print(f"Paso t={t} ({tok}): ||h_{t}|| = {np.linalg.norm(h):.4f} | Probabilidad máxima = {np.max(probs):.2%}")


<IPython.core.display.Image object>
--- EJECUCIÓN PASO A PASO RNN ---
Paso t=1 (el): ||h_1|| = 0.9177 | Probabilidad máxima = 36.97%
Paso t=2 (gato): ||h_2|| = 0.9730 | Probabilidad máxima = 32.77%
Paso t=3 (duerme): ||h_3|| = 1.3977 | Probabilidad máxima = 58.81%


<IPython.core.display.Image object>

### 4.4 Desvanecimiento y Explosión del Gradiente en RNNs
El entrenamiento de una RNN desenrollada se efectúa mediante el algoritmo de **Retropropagación a través del Tiempo (Backpropagation Through Time, BPTT)**.

Si consideramos la pérdida en el paso final $T$ (por ejemplo, predecir una palabra distante), el gradiente respecto al estado oculto inicial $\mathbf{h}_1$ se calcula aplicando la regla de la cadena multivariable:
$$\frac{\partial \mathcal{L}_T}{\partial \mathbf{h}_1} = \frac{\partial \mathcal{L}_T}{\partial \mathbf{h}_T} \cdot \prod_{k=2}^T \frac{\partial \mathbf{h}_k}{\partial \mathbf{h}_{k-1}}$$
donde cada término del producto matricial jacobiano es:
$$\frac{\partial \mathbf{h}_k}{\partial \mathbf{h}_{k-1}} = \text{diag}(1 - \tanh^2(\mathbf{z}_k)) \cdot W_{hh}^T$$

* **Desvanecimiento del Gradiente (Vanishing Gradient):**  
  La derivada de la tangente hiperbólica está estrictamente acotada: $|\tanh'(z)| \le 1$ (siendo menor a 0.5 en la gran mayoría del dominio). Si el mayor autovalor de la matriz de transición recurrente $W_{hh}$ es menor a 1 (o si $\tanh$ satura), el producto acumulado de estas matrices decrece de forma **exponencial**:
  $$\left\| \prod_{k=2}^T \frac{\partial \mathbf{h}_k}{\partial \mathbf{h}_{k-1}} \right\| \propto \gamma^{T-1} \quad (\gamma < 1)$$
  Apenas $T - t \ge 10$, el gradiente se atenúa prácticamente a cero ($0.5^{10} \approx 0.00097$). En consecuencia, los pesos no reciben señal de actualización sobre dependencias lejanas: el modelo "olvida" el contexto remoto y no puede aprender reglas gramaticales de largo alcance (e.g., concordancia entre un sujeto al inicio del párrafo y su verbo conjugado 15 palabras después).
* **Explosión del Gradiente (Exploding Gradient):**  
  Si el mayor autovalor de $W_{hh}$ es significativamente mayor a 1, el producto matricial crece exponencialmente hacia el infinito, provocando oscilaciones violentas, desbordamiento numérico (*overflow*) y pesos con valores `NaN`. Afortunadamente, la explosión se puede controlar mediante la técnica de **Gradient Clipping** (recortar la norma del gradiente a un umbral $c$). Sin embargo, el desvanecimiento del gradiente no se soluciona simplemente con factores de escala, requiriendo un cambio radical en la arquitectura de la celda recurrente.


In [ ]:
# Visualización del Problema del Desvanecimiento del Gradiente
display(Image(filename="Lab#7/figuras/07_desvanecimiento_gradiente.png", width=750))


<IPython.core.display.Image object>


<IPython.core.display.Image object>

## 5. LSTM y GRU: Ventajas y Limitaciones

Las arquitecturas de memoria con compuertas surgieron formalmente para solucionar el desvanecimiento del gradiente en las RNNs simples, permitiendo crear "autopistas" de flujo constante de información.

### 5.1 ¿Qué Problema de las RNN Intentan Resolver las LSTM?
Las **LSTM (Long Short-Term Memory)**, propuestas por Hochreiter y Schmidhuber (1997), resuelven específicamente el desvanecimiento del gradiente. En lugar de forzar que toda la información se comprima y se multiplique repetidamente a través de una función saturante $\tanh$ en cada paso temporal, la LSTM introduce una línea de memoria interna continua (**Cell State**) cuyo flujo hacia adelante y hacia atrás se basa en **operaciones aditivas lineales**, permitiendo que el gradiente fluya intacto a lo largo de cientos de pasos temporales.

---

### 5.2 Celda de Memoria y Compuertas de la LSTM
La celda LSTM controla el flujo de información mediante tres compuertas (*gates*) parametrizadas por funciones sigmoides $\sigma \in (0, 1)$:

1. **Línea de Memoria / Estado de la Celda ($C_t$):**  
   Es la autopista central de información que atraviesa horizontalmente la secuencia. Al experimentar únicamente adiciones y multiplicaciones punto a punto, actúa como un buffer sin atenuación exponencial del gradiente.
2. **Compuerta de Olvido (Forget Gate, $\mathbf{f}_t$):**  
   $$\mathbf{f}_t = \sigma(W_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f)$$
   * **¿Qué decide?:** Examina la nueva entrada $\mathbf{x}_t$ y el estado previo $\mathbf{h}_{t-1}$, y decide qué fracción de la información pasada guardada en $C_{t-1}$ debe ser **descartada** (0 = olvidar por completo, 1 = retener íntegramente). *Ejemplo en NLP:* Si la oración venía hablando de *"Elizabeth"* y aparece un nuevo sujeto *"Mr. Darcy"*, la compuerta de olvido purga el género femenino de la memoria de concordancia.
3. **Compuerta de Entrada (Input Gate, $\mathbf{i}_t$) y Candidato ($\tilde{\mathbf{C}}_t$):**  
   $$\mathbf{i}_t = \sigma(W_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i), \quad \tilde{\mathbf{C}}_t = \tanh(W_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c)$$
   * **¿Qué decide?:** $\mathbf{i}_t$ decide qué componentes del nuevo candidato $\tilde{\mathbf{C}}_t$ vale la pena almacenar en la memoria a largo plazo.
   * **Actualización del Estado de Celda:**  
     $$C_t = \mathbf{f}_t \odot C_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{C}}_t$$
     Nótese la naturaleza **aditiva** de la actualización: si $\mathbf{f}_t \approx 1$ e $\mathbf{i}_t \approx 0$, $C_t = C_{t-1}$, preservando el gradiente exactamente intacto.
4. **Compuerta de Salida (Output Gate, $\mathbf{o}_t$):**  
   $$\mathbf{o}_t = \sigma(W_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o), \quad \mathbf{h}_t = \mathbf{o}_t \odot \tanh(C_t)$$
   * **¿Qué decide?:** Determina qué porción filtrada de la memoria interna $C_t$ debe emerger como el estado oculto visible $\mathbf{h}_t$ para alimentar la salida actual o pasar a la siguiente celda.

---

### 5.3 Diferencias de la GRU (Gated Recurrent Unit) frente a la LSTM
Propuesta por Cho et al. (2014), la **GRU** es una simplificación elegante y computacionalmente eficiente de la LSTM:
* **Fusión de Estados:** Elimina la distinción separada entre Cell State $C_t$ y Hidden State $\mathbf{h}_t$, unificándolos en un único vector de estado $\mathbf{h}_t$.
* **Reducción de Compuertas:** Sustituye las 3 compuertas de la LSTM por únicamente 2 compuertas:
  1. *Compuerta de Reinicio (Reset Gate, $\mathbf{r}_t$):* Decide cuánta memoria pasada recordar al calcular la propuesta de nuevo candidato.
  2. *Compuerta de Actualización (Update Gate, $\mathbf{z}_t$):* Acopla simultáneamente el olvido y la entrada:
     $$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$
* **Ventaja Práctica:** Posee un **25% menos de parámetros** que una LSTM con la misma dimensión oculta. Entrena notablemente más rápido, demanda menos memoria de GPU y generaliza de forma superior en conjuntos de datos medianos o reducidos donde la LSTM tiende a sobreajustar.


In [ ]:
# Visualización de la Comparación Arquitectónica: LSTM vs GRU
display(Image(filename="Lab#7/figuras/08_arquitectura_lstm_vs_gru.png", width=900))


<IPython.core.display.Image object>


<IPython.core.display.Image object>

### 5.4 Tabla Comparativa: RNN vs LSTM vs GRU

A continuación se sintetizan las cuatro dimensiones de análisis requeridas para comparar las tres familias de arquitecturas recurrentes:

| Dimensión de Análisis | Red Recurrente Simple (RNN de Elman) | LSTM (Long Short-Term Memory) | GRU (Gated Recurrent Unit) |
| :--- | :--- | :--- | :--- |
| **Capacidad para capturar dependencias largas** | **Muy baja.** Se degrada severamente tras 8–12 pasos temporales debido al desvanecimiento del gradiente. | **Muy alta.** Capaz de retener dependencias a lo largo de cientos de pasos temporales gracias a la aditividad de su *Cell State*. | **Alta / Muy alta.** Rendimiento prácticamente equivalente a LSTM en secuencias de longitud moderada a larga. |
| **Número de parámetros ($d_h$ oculto, $d_x$ entrada)** | **Mínimo.** $1 \times (d_h^2 + d_h d_x + d_h)$. Una sola matriz de pesos de transición. | **Máximo.** $4 \times (d_h^2 + d_h d_x + d_h)$. Cuatro conjuntos de matrices (3 compuertas + 1 candidato). | **Moderado.** $3 \times (d_h^2 + d_h d_x + d_h)$. Tres conjuntos de matrices (~25% menos parámetros que LSTM). |
| **Velocidad de entrenamiento** | **Muy rápida por época**, pero converge con dificultad y pobremente sobre dependencias complejas. | **Más lenta.** Mayor sobrecarga de cálculo por paso y múltiples operaciones tensoriales por celda. | **Significativamente más rápida que LSTM** (~20% a 30% más rápida por época con convergencia más fluida). |
| **Posibilidad de paralelización** | **Nula en la dimensión temporal.** Complejidad temporal $O(T)$ estrictamente secuencial; $\mathbf{h}_t$ depende de $\mathbf{h}_{t-1}$. | **Nula en la dimensión temporal.** Complejidad temporal $O(T)$ estrictamente secuencial. | **Nula en la dimensión temporal.** Complejidad temporal $O(T)$ estrictamente secuencial. |


### 5.5 Intuición: Limitaciones que Motivaron los Mecanismos de Atención y Transformers
A pesar del éxito rotundo de las LSTM y GRU, dos limitaciones intrínsecas insalvables motivaron el cambio de paradigma hacia la Atención y los Transformers (Vaswani et al., 2017):

1. **Cuello de Botella de Información del Estado Oculto (Information Bottleneck):**  
   En los modelos recurrentes, toda la historia semántica de una oración de entrada de longitud arbitraria debe ser forzada y comprimida dentro de un único vector numérico estático de tamaño fijo $\mathbf{h}_T$. A medida que las oraciones crecen (e.g., documentos legales, narrativas completas), los matices iniciales se diluyen inevitablemente.
2. **Naturaleza Estrictamente Secuencial (No Paralelizable):**  
   Por definición, el paso $t$ no puede calcularse hasta que el paso $t-1$ haya finalizado por completo. Esto genera una dependencia serial temporal $O(T)$ que imposibilita aprovechar el hardware moderno masivamente paralelo (GPUs y TPUs), volviendo prohibitivo el preentrenamiento sobre corpus de miles de millones de tokens.

**La Solución de la Auto-Atención y el Transformer:**  
El mecanismo de Atención elimina la recurrencia. En vez de viajar palabra por palabra arrastrando una memoria, cada token calcula directamente su grado de afinidad con todos los demás tokens de la secuencia en una sola operación de profundidad $O(1)$. Esto permite que la longitud de la ruta entre cualquier par de palabras sea siempre unitaria y habilita la **paralelización total** del cómputo durante el entrenamiento.


## 6. Usos y Aplicaciones en NLP

Los modelos de secuencia se configuran en distintos esquemas de entrada y salida según la naturaleza lingüística de la tarea a resolver.

### 6.1 Esquemas de Modelos de Secuencia
1. **Muchos-a-Uno (Many-to-One):**  
   * **Estructura:** La red recibe una secuencia temporal completa de tokens $\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_T$ y emite una única predicción $\hat{\mathbf{y}}$ derivada del estado oculto final $\mathbf{h}_T$.
   * **Ejemplo de Tarea en NLP:** Clasificación de Sentimiento en reseñas (e.g., determinar si una crítica de cine es positiva o negativa), detección de Spam en correos electrónicos o perfilamiento de autor.
2. **Muchos-a-Muchos Sincrónico / Alineado (Many-to-Many Synced / Sequence Labeling):**  
   * **Estructura:** La entrada es una secuencia de longitud $T$, y la red emite exactamente una etiqueta por cada paso temporal: $\hat{\mathbf{y}}_1, \hat{\mathbf{y}}_2, \dots, \hat{\mathbf{y}}_T$, manteniendo una sincronización estricta token a token.
   * **Ejemplo de Tarea en NLP:** Etiquetado Gramatical (POS Tagging) y Reconocimiento de Entidades Nombradas (NER) bajo el esquema BIO.
3. **Secuencia-a-Secuencia (Seq2Seq / Encoder-Decoder):**  
   * **Estructura:** Una secuencia de entrada de longitud $T_x$ es consumida por un *Encoder* que genera un vector de contexto; luego, un *Decoder* genera de manera autorregresiva una secuencia de salida de longitud variable $T_y$ (donde $T_x \neq T_y$).
   * **Ejemplo de Tarea en NLP:** Traducción Automática de idiomas (e.g., español a inglés), Resumen Automático Abstractivo y generación de diálogo en Chatbots.

---

### 6.2 Relación con Tareas de Laboratorios Anteriores
* **Modelo de Lenguaje para Predicción de la Siguiente Palabra (Laboratorio #5):**  
  Corresponde a un esquema **Muchos-a-Muchos Autorregresivo**. En cada paso temporal $t$, la red recibe el prefijo de palabras previas y predice una distribución categórica sobre el vocabulario para el token inmediatamente siguiente $\mathbf{x}_{t+1}$.
* **Etiquetado POS y Reconocimiento de Entidades NER (Laboratorio #6):**  
  Corresponde con precisión matemática al esquema **Muchos-a-Muchos Sincrónico**. Cada token de las novelas de Austen, Shelley y Stoker recibía una etiqueta gramatical (Penn Treebank/Universal) o una etiqueta de entidad nombrada (e.g., `B-PERSON`, `I-PERSON`, `O`), requiriendo que la longitud de salida coincida exactamente con la longitud de entrada.

---

### 6.3 Aplicación Real en NLP: Traducción Automática Neuronal (NMT)
* **Sistema:** Traducción Automática Neuronal con Arquitectura Seq2Seq y Celdas LSTM.
* **Fuente Académica:**  
  *Sutskever, I., Vinyals, O., & Le, Q. V. (2014). Sequence to Sequence Learning with Neural Networks. Advances in Neural Information Processing Systems (NeurIPS 2014).* Disponible en: [arXiv:1409.3215](https://arxiv.org/abs/1409.3215).
* **Descripción de la Implementación:**  
  Sutskever et al. revolucionaron la traducción automática al reemplazar los engorrosos sistemas estadísticos basados en tablas de frases por una arquitectura unificada de aprendizaje profundo:
  1. **Encoder:** Una LSTM multicapa profunda (4 capas con 1,000 unidades ocultas cada una) lee la oración en el idioma origen (e.g., francés) token por token. Su estado oculto final comprime el significado semántico de toda la oración en un vector continuo.
  2. **Decoder:** Otra LSTM profunda de 4 capas se inicializa con el estado oculto del encoder y genera secuencialmente la traducción en el idioma destino (inglés) token por token, retroalimentando en cada paso la palabra previamente emitida.
  3. **Técnica Innovadora Clave:** Los autores descubrieron que **invertir el orden de la oración en el encoder** (alimentar *"le chat est noir"* como *"noir est chat le"*) reducía sustancialmente la distancia temporal entre las primeras palabras de la oración de origen y las primeras palabras de destino, combatiendo el desvanecimiento del gradiente en el arranque de la traducción y superando por primera vez a los mejores traductores estadísticos de la época en el benchmark WMT'14.


In [ ]:
# Visualización de los Tres Esquemas de Modelos de Secuencia
display(Image(filename="Lab#7/figuras/09_esquemas_modelos_secuencia.png", width=900))


<IPython.core.display.Image object>


<IPython.core.display.Image object>

## 7. Análisis Crítico y Síntesis

Esta sección integra y contrasta de manera crítica los principios examinados a lo largo del laboratorio con los conocimientos adquiridos en las prácticas previas del curso.

### 7.1 Ventana Fija N-grama vs Contexto Recurrente en RNN
* **Diferencia Fundamental en el Modelado del Contexto:**  
  * **N-gramas (Laboratorio #5):** Aplican una hipótesis de Markov de orden rígido $n-1$:
    $$P(w_t | w_1, \dots, w_{t-1}) \approx P(w_t | w_{t-n+1}, \dots, w_{t-1})$$
    El modelo es completamente ciego a cualquier palabra que ocurra antes del límite estricto de la ventana (para un trigrama, todo lo anterior a las dos palabras previas deja de existir).
  * **RNN:** Mantiene un estado latente recursivo $\mathbf{h}_t = f(\mathbf{h}_{t-1}, \mathbf{x}_t)$. Teóricamente, el contexto no tiene un límite rígido prefijado; el estado oculto condensa toda la trayectoria histórica desde el inicio de la secuencia.
* **Ventajas y Desventajas Comparadas:**
  * *N-gramas:*  
    * *Ventajas:* Entrenamiento y evaluación casi instantáneos (conteos de frecuencia); transparencia e interpretabilidad matemática total.
    * *Desventajas:* Explosión combinatoria de parámetros ($|V|^n$); incapacidad para modelar dependencias de mediano y largo alcance; nula generalización semántica.
  * *RNN:*  
    * *Ventajas:* Número de parámetros constante e independiente de la longitud del texto; capacidad para modelar dependencias contextuales fluidas; transferencia semántica mediante representaciones aprendidas.
    * *Desventajas:* Costo de cómputo sustancialmente mayor; lentitud de inferencia secuencial; propensión al desvanecimiento del gradiente en modelos básicos.

---

### 7.2 Embeddings frente a Palabras Fuera de Vocabulario (OOV) y Escasez de Datos
En el **Laboratorio #5 y #6**, constatamos cómo el vocabulario no visto o arcaico (e.g., arcaísmos de Bram Stoker en *Dracula*) provocaba dos fallas críticas:
1. En n-gramas, una sola palabra no observada provocaba la anulación a cero de la probabilidad de la secuencia completa, exigiendo métodos heurísticos de suavizado (Laplace, Good-Turing).
2. En etiquetado gramatical y NER, las palabras fuera de dominio recibían la etiqueta más frecuente en inglés contemporáneo, degradando drásticamente el F1 (cayendo a 0.328 en *Dracula*).

**¿Cómo mitigan este problema las representaciones aprendidas?**
* **Espacio Continuo vs Discreto:** A diferencia de los n-gramas, las representaciones aprendidas mapean el léxico a un espacio continuo. Si una combinación específica de palabras no ocurrió en el entrenamiento, el modelo interpola en el espacio vectorial calculando proyecciones afines a partir de palabras semánticamente vecinas.
* **Embeddings de Subpalabras (Subword Tokenization: BPE, WordPiece, FastText):** En los modelos neuronales modernos, las palabras OOV ya no se asignan a un token ciego `<UNK>`. El vocabulario se construye a nivel de morfemas y sub-palabras. Por ejemplo, una palabra no vista como *"ultrarrápido"* se segmenta en `["ultra", "##rápido"]`. Los embeddings de los morfemas retienen la semántica básica y la red compone vectorialmente el significado exacto de la palabra desconocida, erradicando por completo el problema del OOV.

---

### 7.3 Elección Justificada: Clasificación de Sentimiento en Reseñas Cortas con Cómputo Limitado
**Escenario:** Clasificar el sentimiento (positivo/negativo) de reseñas de una o dos oraciones (longitud típica de 15 a 35 tokens), disponiendo de recursos de cómputo estrictamente limitados (CPU estándar o GPU de baja gama).

**Elección Técnica Recomendada:** **GRU (Gated Recurrent Unit)**.

**Justificación Técnica Detallada:**
1. **¿Por qué descartar la RNN simple?:** A pesar de que las oraciones son cortas, la RNN simple sufre de una superficie de pérdida muy irregular con valles abruptos que dificultan la convergencia estable. Además, en análisis de sentimiento el significado suele invertirse mediante modificadores de negación al inicio de la frase (*"No creo que sea la mejor película..."*); una RNN simple puede atenuar el impacto del *"No"* antes de llegar a la predicción final.
2. **¿Por qué GRU sobre LSTM?:**
   * *Ahorro de Parámetros:* La LSTM utiliza 4 conjuntos de matrices de pesos por celda frente a los 3 conjuntos de la GRU (la GRU tiene un **25% menos de parámetros entrenables**).
   * *Velocidad y Memoria:* Con recursos de cómputo limitados, la GRU entrena aproximadamente entre un **20% y 30% más rápido por época** y consume significativamente menos memoria RAM/VRAM en GPU.
   * *Adecuación al Dominio:* La gran ventaja de la LSTM (su *Cell State* para recordar cientos de pasos temporales) es redundante en secuencias de apenas 20 tokens. Para textos cortos, la GRU ofrece la misma capacidad de representación que la LSTM con menor riesgo de sobreajuste y una convergencia mucho más ágil.

---

### 7.4 Pregunta Abierta de Investigación para la Clase Introductoria
Durante la investigación sobre las limitaciones de las redes recurrentes y la emergencia de los mecanismos de auto-atención, surge una interrogante crucial sobre el estado actual del arte:

> *"Si las arquitecturas recurrentes con compuertas (LSTM/GRU) lograron estabilizar el gradiente pero fueron desplazadas por los Transformers debido a su cuello de botella estrictamente secuencial $O(T)$, ¿por qué hoy en día la complejidad computacional y de memoria cuadrática $O(N^2)$ de la auto-atención está motivando el surgimiento de nuevos modelos lineales y de espacio de estados como Mamba (SSMs)? ¿En qué se diferencian matemáticamente estas nuevas arquitecturas recurrentes selectivas para lograr que el entrenamiento sea masivamente paralelizable sin perder la capacidad de capturar dependencias de contexto ultra-largo?"*
